In [11]:
import os
import sys
import json
import numpy as np
from pathlib import Path
from datetime import datetime

import psycopg2
from psycopg2.extras import Json, execute_values
from dotenv import load_dotenv
from IPython.display import display, Markdown

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import (
    TEXT_CHUNKS_PATH,
    EMBEDDINGS_NPY_PATH,
    EMBEDDINGS_META_PATH,
    FILTER_MANIFEST_PATH,
    CHUNK_IMAGE_LINKS_PATH,
    FILTERED_KEEP_DIR,
    DOCUMENT_NAME,
    EMBEDDING_DIMENSION,
    ENV_PATH,
    OUTPUT_DIRS,
)

for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH)
PG_CONN_STR = os.getenv("PG_CONN_STR")
assert PG_CONN_STR, "PG_CONN_STR not found in .env"

display(Markdown("### Environment loaded"))

### Environment loaded

In [12]:
conn = psycopg2.connect(PG_CONN_STR)
cur = conn.cursor()

# Check pgvector extension
cur.execute("SELECT extname FROM pg_extension WHERE extname = 'vector';")
pgvector_ok = cur.fetchone() is not None

# Check tables exist
required_tables = ["chunks", "images", "chunk_images"]
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' AND table_name = ANY(%s);
""", (required_tables,))
found_tables = {row[0] for row in cur.fetchall()}
missing_tables = set(required_tables) - found_tables

cur.close()
conn.close()

table_rows = "\n".join(
    f"| `{t}` | {'Found' if t in found_tables else 'MISSING'} |"
    for t in required_tables
)

display(Markdown(f"""
### Database Connection Verified

| Item | Value |
|------|-------|
| **pgvector extension** | {'Yes' if pgvector_ok else 'MISSING'} |

| Table | Status |
|-------|--------|
{table_rows}
""".strip()))

assert pgvector_ok, "pgvector extension not installed"
assert not missing_tables, f"Missing tables: {missing_tables}"

### Database Connection Verified

| Item | Value |
|------|-------|
| **pgvector extension** | Yes |

| Table | Status |
|-------|--------|
| `chunks` | Found |
| `images` | Found |
| `chunk_images` | Found |

In [13]:
chunks = json.loads(TEXT_CHUNKS_PATH.read_text(encoding="utf-8"))
embeddings = np.load(str(EMBEDDINGS_NPY_PATH))
metadata = json.loads(EMBEDDINGS_META_PATH.read_text(encoding="utf-8"))
filter_manifest = json.loads(FILTER_MANIFEST_PATH.read_text(encoding="utf-8"))
chunk_image_links = json.loads(CHUNK_IMAGE_LINKS_PATH.read_text(encoding="utf-8"))

kept_images = [img for img in filter_manifest if img.get("decision") == "keep"]

# Validations
assert embeddings.shape[0] == len(chunks), (
    f"Mismatch: {embeddings.shape[0]} embeddings vs {len(chunks)} chunks"
)
assert embeddings.shape[1] == EMBEDDING_DIMENSION, (
    f"Expected dim {EMBEDDING_DIMENSION}, got {embeddings.shape[1]}"
)
assert len(metadata) == len(chunks), (
    f"Mismatch: {len(metadata)} metadata vs {len(chunks)} chunks"
)

display(Markdown(f"""
### Local Files Loaded

| Item | Count |
|------|-------|
| **Chunks** | {len(chunks)} |
| **Embeddings** | {embeddings.shape[0]} x {embeddings.shape[1]} |
| **Metadata** | {len(metadata)} |
| **Kept images** | {len(kept_images)} |
| **Chunk-image links** | {len(chunk_image_links)} |
""".strip()))

### Local Files Loaded

| Item | Count |
|------|-------|
| **Chunks** | 1391 |
| **Embeddings** | 1391 x 768 |
| **Metadata** | 1391 |
| **Kept images** | 866 |
| **Chunk-image links** | 1391 |

In [14]:
conn = psycopg2.connect(PG_CONN_STR)
cur = conn.cursor()

insert_sql = """
    INSERT INTO chunks (
        chunk_id, chunk_index, document_name, text,
        page_start, page_end, source_pages,
        toc_level, section_title, section_path,
        char_count, word_count, embedding
    ) VALUES (
        %s, %s, %s, %s,
        %s, %s, %s::jsonb,
        %s, %s, %s::jsonb,
        %s, %s, %s::vector
    )
    ON CONFLICT (chunk_id) DO UPDATE SET
        chunk_index = EXCLUDED.chunk_index,
        document_name = EXCLUDED.document_name,
        text = EXCLUDED.text,
        page_start = EXCLUDED.page_start,
        page_end = EXCLUDED.page_end,
        source_pages = EXCLUDED.source_pages,
        toc_level = EXCLUDED.toc_level,
        section_title = EXCLUDED.section_title,
        section_path = EXCLUDED.section_path,
        char_count = EXCLUDED.char_count,
        word_count = EXCLUDED.word_count,
        embedding = EXCLUDED.embedding;
"""

rows_affected = 0
for i, chunk in enumerate(chunks):
    emb_list = embeddings[i].tolist()
    cur.execute(insert_sql, (
        chunk["chunk_id"],
        chunk["chunk_index"],
        DOCUMENT_NAME,
        chunk["text"],
        chunk["page_start"],
        chunk["page_end"],
        json.dumps(chunk["source_pages"]),
        chunk["toc_level"],
        chunk["section_title"],
        json.dumps(chunk["section_path"]),
        chunk["char_count"],
        chunk["word_count"],
        str(emb_list),
    ))
    rows_affected += 1

conn.commit()
cur.close()
conn.close()

display(Markdown(f"""
### Chunks Inserted

| Item | Value |
|------|-------|
| **Rows inserted/updated** | {rows_affected} |
""".strip()))

### Chunks Inserted

| Item | Value |
|------|-------|
| **Rows inserted/updated** | 1391 |

In [15]:
conn = psycopg2.connect(PG_CONN_STR)
cur = conn.cursor()

insert_img_sql = """
    INSERT INTO images (
        filename, document_name, page,
        width, height, file_size_bytes,
        image_path, md5, decision
    ) VALUES (
        %s, %s, %s,
        %s, %s, %s,
        %s, %s, %s
    )
    ON CONFLICT (filename) DO UPDATE SET
        document_name = EXCLUDED.document_name,
        page = EXCLUDED.page,
        width = EXCLUDED.width,
        height = EXCLUDED.height,
        file_size_bytes = EXCLUDED.file_size_bytes,
        image_path = EXCLUDED.image_path,
        md5 = EXCLUDED.md5,
        decision = EXCLUDED.decision;
"""

img_rows = 0
for img in kept_images:
    rel_path = f"data/filtered/keep/{img['filename']}"
    cur.execute(insert_img_sql, (
        img["filename"],
        DOCUMENT_NAME,
        img["page"],
        img["width"],
        img["height"],
        img["file_size_bytes"],
        rel_path,
        img.get("md5", ""),
        "keep",
    ))
    img_rows += 1

conn.commit()
cur.close()
conn.close()

display(Markdown(f"""
### Images Inserted

| Item | Value |
|------|-------|
| **Rows inserted/updated** | {img_rows} |
""".strip()))

### Images Inserted

| Item | Value |
|------|-------|
| **Rows inserted/updated** | 866 |

In [16]:
conn = psycopg2.connect(PG_CONN_STR)
cur = conn.cursor()

insert_link_sql = """
    INSERT INTO chunk_images (
        chunk_id, image_filename, link_reason
    ) VALUES (
        %s, %s, %s
    )
    ON CONFLICT (chunk_id, image_filename) DO NOTHING;
"""

link_rows = 0
for record in chunk_image_links:
    if record["image_count"] == 0:
        continue
    for img in record["images"]:
        cur.execute(insert_link_sql, (
            record["chunk_id"],
            img["filename"],
            "page_overlap",
        ))
        link_rows += cur.rowcount

conn.commit()
cur.close()
conn.close()

display(Markdown(f"""
### Chunk-Image Links Inserted

| Item | Value |
|------|-------|
| **Rows inserted** | {link_rows} |
""".strip()))

### Chunk-Image Links Inserted

| Item | Value |
|------|-------|
| **Rows inserted** | 0 |

In [17]:
conn = psycopg2.connect(PG_CONN_STR)
cur = conn.cursor()

counts = {}
for table in ["chunks", "images", "chunk_images"]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    counts[table] = cur.fetchone()[0]

# Correct pgvector dimension check
cur.execute("""
    SELECT vector_dims(embedding)
    FROM chunks
    WHERE embedding IS NOT NULL
    LIMIT 1;
""")
db_dim = cur.fetchone()[0]

# Extra safety checks
cur.execute("""
    SELECT COUNT(*)
    FROM chunks
    WHERE embedding IS NOT NULL;
""")
chunks_with_embeddings = cur.fetchone()[0]

cur.close()
conn.close()

count_rows = "\n".join(f"| `{t}` | {c} |" for t, c in counts.items())

display(Markdown(f"""
### Database Validation

| Table | Row Count |
|-------|-----------|
{count_rows}

| Check | Value |
|-------|-------|
| **Chunks with embeddings** | {chunks_with_embeddings} |
| **Embedding dimension in DB** | {db_dim} |
| **Expected embedding dimension** | {EMBEDDING_DIMENSION} |
""".strip()))

assert db_dim == EMBEDDING_DIMENSION, (
    f"Embedding dimension mismatch: DB={db_dim}, expected={EMBEDDING_DIMENSION}"
)

assert chunks_with_embeddings == counts["chunks"], (
    f"Some chunks are missing embeddings: {chunks_with_embeddings}/{counts['chunks']}"
)

### Database Validation

| Table | Row Count |
|-------|-----------|
| `chunks` | 1391 |
| `images` | 866 |
| `chunk_images` | 1282 |

| Check | Value |
|-------|-------|
| **Chunks with embeddings** | 1391 |
| **Embedding dimension in DB** | 768 |
| **Expected embedding dimension** | 768 |

In [18]:
display(Markdown(f"""
---

## Notebook 07 Complete — Database Ingestion

| Item | Value |
|------|-------|
| **Database** | `scada_dip_db` |
| **Chunks** | {counts['chunks']} rows |
| **Images** | {counts['images']} rows |
| **Chunk-image links** | {counts['chunk_images']} rows |
| **Embedding model** | `sentence-transformers/all-mpnet-base-v2` |
| **Embedding dimension** | {EMBEDDING_DIMENSION} |
| **Upsert safe** | Yes (ON CONFLICT) |
| **Timestamp** | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} |

All prepared data is now in PostgreSQL with pgvector indexing.
""".strip()))

---

## Notebook 07 Complete — Database Ingestion

| Item | Value |
|------|-------|
| **Database** | `scada_dip_db` |
| **Chunks** | 1391 rows |
| **Images** | 866 rows |
| **Chunk-image links** | 1282 rows |
| **Embedding model** | `sentence-transformers/all-mpnet-base-v2` |
| **Embedding dimension** | 768 |
| **Upsert safe** | Yes (ON CONFLICT) |
| **Timestamp** | 2026-04-24 11:07:54 |

All prepared data is now in PostgreSQL with pgvector indexing.